# VocEd Lab 10 — End to End: the N/C Ratio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emilsar/VocEd/blob/main/Labs/Lab10_end_to_end_nc_ratio.ipynb)

The whole course has been about one number: the **nucleus-to-cytoplasm (N/C) ratio**, which a
pathologist uses as a sign that a cell may be cancerous.  We mostly scored our models with Dice,
because Dice is what segmentation is judged by.  In this last lab we judge them by the N/C ratio
itself, the way a user of the tool would:

- the best model of each family — **thresholds + clean-up** (Lab 4), **random forest + features**
  (Lab 7), **U-Net** (Lab 9) — on the same 40 test images;
- how close each one's N/C ratio is, whether it is **biased**, and **where** it fails;
- what happens when the N/C ratio is used to **flag** images for a closer look.

**Textbook:** [§4.7 Comparing All Four Stages](https://obooks.tech/imageprocessing/chapter4.html#comparing-all-four-stages),
[Chapter 8 — Evaluation and Failure Modes](https://obooks.tech/imageprocessing/chapter8.html#evaluation-and-failure-modes), and
[Appendix C — Evaluation Metrics](https://obooks.tech/imageprocessing/appendixc.html#sec-eval-metrics).

---

This lab runs fine on a CPU: the U-Net is not retrained, its weights come with the repository.

### How the puzzles work 🧩🤔

- **🧩 Code puzzles:** replace every `___` and run the cell — a ✅ means you got it.
- **🤔 Think puzzles:** answer in the ✏️ cell below the question.  There is no check: discuss
  your answer with your group.

## 0. Setup — Get the Data

The same images and split as Labs 3–9.

In [ ]:
!git clone https://github.com/emilsar/VocEd.git
%cd VocEd

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

X = np.stack([np.load(f"imagedata/X/{i}.npy") for i in range(200)])   # (200, 3, 256, 256)
y = np.stack([np.load(f"imagedata/y/{i}.npy") for i in range(200)])   # (200, 256, 256)
gray_all = 0.299 * X[:, 0] + 0.587 * X[:, 1] + 0.114 * X[:, 2]

nuc_px   = (y == 2).sum(axis=(1, 2))
quartile = np.digitize(nuc_px, np.percentile(nuc_px, [25, 50, 75]))
trainval_idx, test_idx = train_test_split(np.arange(200), test_size=0.2, stratify=quartile, random_state=42)
train_idx, val_idx = train_test_split(trainval_idx, test_size=40, stratify=quartile[trainval_idx], random_state=42)
assert list(test_idx[:5]) == [121, 152, 81, 4, 141], "Not the same test images as Labs 2–9"

def dice_score(pred, target, label):
    pred_mask   = (pred   == label)
    target_mask = (target == label)
    denominator = pred_mask.sum() + target_mask.sum()
    if denominator == 0:
        return 1.0
    return 2 * (pred_mask & target_mask).sum() / denominator

def image_dice(pred, mask):
    return (dice_score(pred, mask, 1) + dice_score(pred, mask, 2)) / 2      # cytoplasm and nucleus

def nc_ratio(mask):
    return (mask == 2).sum() / max((mask == 1).sum(), 1)

## 1. Three Models, Rebuilt

Each model is rebuilt from its lab in a few lines.  Nothing here is tuned again: the settings are
the ones each lab chose on its **validation** images.  *(Under a minute, mostly the random forest.)*
Lab 4's thresholds are rounded to three decimals here, so its numbers can differ from Lab 4's in the
third decimal.

In [ ]:
from skimage.measure import label
from scipy.ndimage import binary_fill_holes, gaussian_filter, median_filter, sobel, uniform_filter
from skimage.filters import gabor
from sklearn.ensemble import RandomForestClassifier

# ── Lab 4: thresholds + remove_specks + fill_holes ──
def segment(gray, t_nucleus, t_background):
    pred = np.zeros(gray.shape, dtype=np.int64)
    pred[gray < t_nucleus] = 2
    pred[(gray >= t_nucleus) & (gray < t_background)] = 1
    return pred

def remove_specks(pred, min_size=100):
    labels    = label(pred == 2, connectivity=2)
    areas     = np.bincount(labels.ravel())
    small_ids = np.where(areas < min_size)[0]
    out = pred.copy()
    out[np.isin(labels, small_ids) & (labels > 0)] = 1
    return out

def fill_holes(pred):
    filled = binary_fill_holes(pred == 2)
    out = pred.copy()
    out[filled & (pred == 1)] = 2
    return out

def lab4_model(i):
    return fill_holes(remove_specks(segment(gray_all[i], 0.353, 1.00)))

# ── Lab 7: random forest on ten feature maps ──
names = ['R', 'G', 'B', 'blur2', 'blur8', 'median5', 'sobel', 'std5', 'gabor0', 'gabor90']

def feature_table(img):
    gray   = 0.299 * img[0] + 0.587 * img[1] + 0.114 * img[2]
    smooth = gaussian_filter(gray, 1)
    maps = [img[0], img[1], img[2], gaussian_filter(gray, 2), gaussian_filter(gray, 8), median_filter(gray, 5),
            np.hypot(sobel(smooth, axis=0), sobel(smooth, axis=1)),
            np.sqrt(np.maximum(uniform_filter(gray ** 2, 5) - uniform_filter(gray, 5) ** 2, 0)),
            gabor(gray, frequency=0.25, theta=0)[0], gabor(gray, frequency=0.25, theta=np.pi / 2)[0]]
    return np.stack([m.ravel() for m in maps], axis=1)

rng = np.random.default_rng(0)
chosen = {i: rng.choice(65536, 450, replace=False) for i in train_idx}
X_tab = np.vstack([feature_table(X[i])[chosen[i]] for i in train_idx])
y_tab = np.concatenate([y[i].reshape(-1)[chosen[i]] for i in train_idx])
forest = RandomForestClassifier(n_estimators=100, min_samples_leaf=100, random_state=0).fit(X_tab, y_tab)

def lab7_model(i):
    return forest.predict(feature_table(X[i])).reshape(256, 256)

print("Lab 4 and Lab 7 models ready.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ── Lab 9: the U-Net, with weights trained by Lab 9's code ──
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, kernel_size=3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class SmallUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = ConvBlock(3, 32)
        self.enc2 = ConvBlock(32, 64)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(64, 128)
        self.dec2 = ConvBlock(128 + 64, 64)
        self.dec1 = ConvBlock(64 + 32, 32)
        self.out_conv = nn.Conv2d(32, 3, kernel_size=1)

    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(self.pool(s1))
        b  = self.bottleneck(self.pool(s2))
        u2 = self.dec2(torch.cat([F.interpolate(b,  scale_factor=2, mode='bilinear', align_corners=False), s2], dim=1))
        u1 = self.dec1(torch.cat([F.interpolate(u2, scale_factor=2, mode='bilinear', align_corners=False), s1], dim=1))
        return self.out_conv(u1)

unet = SmallUNet()
unet.load_state_dict(torch.load('models/unet_lab9.pt', map_location='cpu'))
unet.eval()

def lab9_model(i):
    with torch.no_grad():
        return unet(torch.from_numpy(X[i:i + 1]))[0].argmax(dim=0).numpy()

print("U-Net ready.")

Now every model segments every test image, and we compute Dice and the N/C ratio.

In [ ]:
models = {'Lab 4 — thresholds + clean-up': lab4_model,
          'Lab 7 — forest + features':     lab7_model,
          'Lab 9 — U-Net':                 lab9_model}

preds   = {name: [model(i) for i in test_idx] for name, model in models.items()}
nc_true = np.array([nc_ratio(y[i]) for i in test_idx])
nc_pred = {name: np.array([nc_ratio(p) for p in preds[name]]) for name in models}
dice    = {name: np.array([image_dice(p, y[i]) for p, i in zip(preds[name], test_idx)]) for name in models}

for name in models:
    print(f"{name:32s} test Dice {dice[name].mean():.4f}")

## 2. How Close Is the N/C Ratio?

Three ways to score the N/C ratio, from the most technical to the most practical:

- **R²** — 1 means perfect; 0 means no better than always guessing the average; below 0, worse;
- **bias** — the average of (predicted − true): does the model *systematically* over- or
  under-estimate?
- **within ±0.05** — the share of images whose N/C ratio is nearly right.

### 🧩 Puzzle 1 — R², bias and "within ±0.05"

📖 Book: [Appendix C — Evaluation Metrics](https://obooks.tech/imageprocessing/appendixc.html#sec-eval-metrics)

<details><summary><b>💡 Hint 1</b> (click to open)</summary>

R² compares the model's squared errors with the squared distances of the true values from their **average**.

</details>

<details><summary><b>💡 Hint 2</b> (click to open)</summary>

Bias is the average *signed* error — no absolute value.  'Within' uses the absolute error.

</details>

In [ ]:
def r_squared(pred, true):
    return 1 - np.sum((pred - true) ** 2) / np.sum((true - true.___) ** 2)

print(f"{'':32s}    R²     bias   within ±0.05")
for name in models:
    err    = nc_pred[name] - nc_true
    bias   = np.___(err)
    within = np.mean(np.___(err) < 0.05)
    print(f"{name:32s} {r_squared(nc_pred[name], nc_true):6.3f}  {bias:+.3f}   {within:.0%}")

# ── Check ──
assert abs(r_squared(nc_true, nc_true) - 1) < 1e-12, "A perfect prediction has R² = 1"
assert abs(r_squared(np.full(40, nc_true.mean()), nc_true)) < 1e-12, "Always guessing the average gives R² = 0"
print("✅ Puzzle 1 solved!")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, name in zip(axes, models):
    ax.scatter(nc_true, nc_pred[name], alpha=0.7)
    ax.plot([0, 5.5], [0, 5.5], 'r--')
    ax.set_xlim(0, 5.5)
    ax.set_ylim(0, 5.5)
    ax.set_xlabel('true N/C ratio')
    ax.set_ylabel('predicted N/C ratio')
    ax.set_title(f'{name}\nR² = {r_squared(nc_pred[name], nc_true):.3f}')
plt.show()

### 🤔 Think 1 — Dice and N/C disagree

1. Rank the three models by test Dice, then by N/C R².  Is it the same order?
2. Lab 4 and Lab 7 had a test Dice near 0.77, yet their N/C R² was poor.  How can a mask be 77%
   right and its N/C ratio still be badly off?  *(Think about what one fraction does when a few
   thousand pixels move from its bottom to its top.)*
3. What does the sign of each model's **bias** tell you?  Which kind of mistake — calling cytoplasm
   nucleus, or the reverse — would make the bias positive?

📖 Book: [§4.7 Comparing All Four Stages](https://obooks.tech/imageprocessing/chapter4.html#comparing-all-four-stages)

✏️ **Your answer:** *(double-click this cell and type here)*

## 3. Where Does It Fail?

### 🧩 Puzzle 2 — The three worst images

Find the three test images where the U-Net's N/C ratio is furthest from the truth.

<details><summary><b>💡 Hint</b> (click to open)</summary>

`np.argsort` sorts from smallest to largest; the last three positions are the largest errors.

</details>

In [ ]:
u_err = np.abs(nc_pred['Lab 9 — U-Net'] - nc_true)
worst = np.argsort(u_err)[___]

fig, axes = plt.subplots(3, 3, figsize=(13, 13))
for row, k in enumerate(worst):
    i = test_idx[k]
    axes[row, 0].imshow(X[i].transpose(1, 2, 0))
    axes[row, 0].set_title(f'test image {i}')
    axes[row, 1].imshow(y[i], cmap='viridis', vmin=0, vmax=2)
    axes[row, 1].set_title(f'truth: N/C {nc_true[k]:.2f}')
    axes[row, 2].imshow(preds['Lab 9 — U-Net'][k], cmap='viridis', vmin=0, vmax=2)
    axes[row, 2].set_title(f'U-Net: N/C {nc_pred["Lab 9 — U-Net"][k]:.2f}, Dice {dice["Lab 9 — U-Net"][k]:.2f}')
for ax in axes.flat:
    ax.axis('off')
plt.show()

# ── Check ──
assert set(worst) == set(np.argsort(u_err)[-3:]) and u_err[worst].min() >= np.sort(u_err)[-3], "The three largest errors"
print("✅ Puzzle 2 solved!")

### 🤔 Think 2 — Reading the failures

1. For each of the three images, what went wrong: nucleus missed, cytoplasm called nucleus, or very
   little cytoplasm to begin with?
2. Is a big N/C error always a low Dice?  Find a case where the Dice is fine but the N/C is far off.
3. If you could add 50 labelled images to the training set, what kind of images would you pick, based
   on these failures?

📖 Book: [§8.4.4 Evaluation and Failure Modes](https://obooks.tech/imageprocessing/chapter8.html#evaluation-and-failure-modes)

✏️ **Your answer:** *(double-click this cell and type here)*

## 4. Using the N/C Ratio to Flag Images

Suppose the tool **flags** an image for a pathologist to look at whenever its N/C ratio is at
least a cutoff.  We use **0.5** as an *illustrative* cutoff — it is not a clinical rule, and our N/C
(nucleus ÷ cytoplasm area of the whole image) is not exactly the ratio pathologists estimate per
cell.  We compare each model's flags with the flags the true masks would give.

- **sensitivity** — of the images that *should* be flagged, the share the model flags;
- **specificity** — of the images that should *not* be flagged, the share it leaves alone.

### 🧩 Puzzle 3 — Sensitivity and specificity

📖 Book: [Appendix C — General Machine Learning Metrics](https://obooks.tech/imageprocessing/appendixc.html#general-machine-learning-metrics)

<details><summary><b>💡 Hint 1</b> (click to open)</summary>

`should` and `flagged` are True/False arrays.  `flagged[should]` keeps the model's flags for only the images that should be flagged.

</details>

<details><summary><b>💡 Hint 2</b> (click to open)</summary>

For specificity, look at the images that should *not* be flagged (`~should`) and count those the model did not flag (`~flagged`).

</details>

In [ ]:
cutoff = 0.5
should = nc_true >= cutoff
print(f"Images that should be flagged: {should.sum()} of 40")
print()
print(f"{'':32s} flagged   sensitivity   specificity")
for name in models:
    flagged = nc_pred[name] >= cutoff
    sensitivity = flagged[___].mean()
    specificity = (~flagged)[___].mean()
    print(f"{name:32s} {flagged.sum():5d}      {sensitivity:6.0%}        {specificity:6.0%}")

# ── Check ──
assert sensitivity == ((nc_pred[name] >= cutoff) & should).sum() / should.sum(), "Of the images that should be flagged, the share flagged"
assert specificity == (~(nc_pred[name] >= cutoff) & ~should).sum() / (~should).sum(), "Of the images that should not be flagged, the share left alone"
print("✅ Puzzle 3 solved!")

### 🤔 Think 3 — Which mistakes matter?

1. For a screening tool, which is worse: an image that should be flagged but is not (a **false
   negative**), or an image flagged that did not need it (a **false positive**)?  Why?
2. If you had to raise one model's sensitivity, would you move the cutoff up or down?  What would it
   cost?
3. Given what you have seen in this lab, would you let any of these models decide alone, without a
   pathologist?  What would you need to see before trusting one more?

📖 Book: [§8.4.5 Deployment Considerations](https://obooks.tech/imageprocessing/chapter8.html#deployment-considerations)

✏️ **Your answer:** *(double-click this cell and type here)*

## 5. The Whole Course on One Page

Every result in the course, on the same 40 test images (earlier rows from the labs' answer-key runs).

In [ ]:
print(f"{'':36s} test Dice   N/C R²")
print(f"{'Lab 2  — threshold (0.40, 0.85), blur':36s}   0.719")
print(f"{'Lab 3  — five-knob pipeline':36s}   0.770    -0.812")
print(f"{'Lab 4  — thresholds + clean-up':36s}   {dice['Lab 4 — thresholds + clean-up'].mean():.3f}   {r_squared(nc_pred['Lab 4 — thresholds + clean-up'], nc_true):7.3f}")
print(f"{'Lab 5  — k-NN, k = 25 (colour)':36s}   0.763    -0.341")
print(f"{'Lab 6  — MLP (colour)':36s}   0.768    -0.467")
print(f"{'Lab 7  — forest + ten features':36s}   {dice['Lab 7 — forest + features'].mean():.3f}   {r_squared(nc_pred['Lab 7 — forest + features'], nc_true):7.3f}")
print(f"{'Lab 8  — MinimalCNN':36s}   0.755    -0.714")
print(f"{'Lab 9  — U-Net':36s}   {dice['Lab 9 — U-Net'].mean():.3f}   {r_squared(nc_pred['Lab 9 — U-Net'], nc_true):7.3f}")

### 🤔 Think 4 — Looking back

1. Which single step across the ten labs improved test Dice the most?  Which improved the N/C ratio
   the most?
2. Dice was our guide for nine labs, but the N/C ratio is the goal.  If you started the course again,
   would you tune on Dice, on N/C error, or on both?  What might change?
3. Every model in this course was chosen on validation and tested once.  Why does that discipline
   matter even more when the number at the end is a medical one?

📖 Book: [Appendix C — Data Leakage](https://obooks.tech/imageprocessing/appendixc.html#data-leakage)

✏️ **Your answer:** *(double-click this cell and type here)*

## Wrap-up — the Course

**Key takeaways:**
- **Score what you care about.**  Dice and the N/C ratio can rank models differently; a ratio
  amplifies the mistakes that move pixels between its top and bottom.
- **Look for bias and for failures,** not only averages.  A few cells with darkly stained cytoplasm
  dominate the N/C error, for every model.
- **A cutoff turns a number into a decision,** and the decision has its own errors: sensitivity and
  specificity, traded against each other by moving the cutoff.
- **The arc of the course:** one brightness value and a rule (Labs 2–4) → three colours and a learned
  rule (5–6) → hand-made neighbourhood features (7) → learned kernels (8) → a learned multi-scale view
  (9).  Each step was tested on validation and scored once on test.

---

### Group discussion 👥

| Person | Change, then re-run Section 4 |
|---|---|
| A | `cutoff = 0.3` |
| B | `cutoff = 0.8` |
| C | Count how many test images have N/C ≥ 0.5 according to *each* model, and compare with the truth |

1. How do sensitivity and specificity trade off as the cutoff moves?
2. Which model would you choose for screening, and at which cutoff?
3. What is the one thing you would want to try next — a new model, more data, or a different way to
   measure the N/C ratio?